In [17]:
import sys
sys.path.append('/home/galk/LanguageDynamics/src') 
from data_generation.tiny_memory_generation import *
from data_generation.tokens_dataset import *
import random
import numpy as np
from tqdm import tqdm

In [ ]:
### Hyperparameters
E = 1
D = 1
N = 4
M = 2
G = 8
L_E = [1 for _ in range(E)]
L_D = [1 for _ in range(D)]
L_N = [3, 4, 5, 6]
L_M = [1 for _ in range(M)]

special_tokens = ['<BOS>', '<EOS>', '<PAD>']

G_MIN = len(special_tokens) + E + D + M  # special tokens, plus other tokens
G_MAX = G_MIN + G - 1

memory_limit = float("inf")

### Sample noise trajectories
noise_list = []
for n_i in range(N):
    noise_list.append(random.sample(range(G_MIN, G_MAX+1), L_N[n_i]))

vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"G{g+1}" for g in range(G)]
token2id = {tok: idx for idx, tok in enumerate(vocab)}
id2token = {idx: tok for tok, idx in token2id.items()}
vocab_size = len(vocab)

# Non-Terminal (NT) vocab
NT_vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"N{n+1}" for n in range(N)]
NT_token2id = {tok: idx for idx, tok in enumerate(NT_vocab)}
NT_id2token = {idx: tok for tok, idx in NT_token2id.items()}
NT_vocab_size = len(NT_vocab)

# Initialize NT to T transitions
NT_to_T = {token: token if token not in [f"N{n+1}" for n in range(N)] else [id2token[id] for id in noise_list[[f"N{n+1}" for n in range(N)].index(token)]] for token in NT_vocab }

### Initialize Transition matrices
P_transitions = np.zeros((E+1, NT_vocab_size, NT_vocab_size))  # [states, source, target]

# zero mode - no memory
P_transitions[0, NT_token2id['<EOS>'], NT_token2id['<BOS>']] = 1
# P_transitions[0, NT_token2id['<BOS>'], NT_token2id['E1']] = 1/(2*N)
P_transitions[0, NT_token2id['<BOS>'], NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/N
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(2*N)
P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['<EOS>']] = 0.2
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['<EOS>']] = 1/(2*N)
np.fill_diagonal(P_transitions[0], 0)

# memory mode
# P_transitions[1, NT_token2id['D1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M  ## THIS WILL BE DECIDED BY THE CONTEXT (which M was memorized)
P_transitions[1, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
P_transitions[1, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 2/(N)
np.fill_diagonal(P_transitions[1], 0)


# Normalize Matrices
P_transitions = P_transitions / P_transitions.sum(axis=-1, keepdims=True) # normalize rows to sum to 1
P_transitions = np.nan_to_num(P_transitions) # replace NaNs with 0


/tmp/ipykernel_1376410/4085610307.py:61: RuntimeWarning: invalid value encountered in divide
  P_transitions = P_transitions / P_transitions.sum(axis=-1, keepdims=True) # normalize rows to sum to 1


In [30]:
max_steps = float('inf')
n_train = 100000
n_val = 10000
train_dataset, train_dataset_NT, train_seen = generate_tiny_memory_dataset(n_samples=n_train, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=max_steps, memory_limit=memory_limit)
val_dataset, val_dataset_NT, val_seen = generate_tiny_memory_dataset(n_samples=n_val, seen=train_seen, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=max_steps, memory_limit=memory_limit)
context_window = 32
train_blocks = prepare_blocks(train_dataset_NT, NT_token2id, context_window)
val_blocks = prepare_blocks(val_dataset_NT, NT_token2id, context_window)

100%|██████████| 10000/10000 [00:01<00:00, 7815.79it/s]


In [29]:
train_dataset_NT[0]

['<BOS>',
 'N2',
 'N1',
 'N3',
 'N4',
 'N3',
 'E1',
 'M1',
 'N2',
 'N4',
 'N3',
 'N4',
 'N2',
 'D1',
 'M1',
 'N2',
 'E1',
 'M1',
 'N4',
 'N1',
 'N2',
 'D1',
 'M1',
 'N1',
 'E1',
 'M2',
 'N1',
 'N2',
 'N1',
 'D1',
 'M2',
 'N1',
 'N3',
 'N1',
 'N3',
 'E1',
 'M2',
 'N4',
 'D1',
 'M2',
 'N1',
 'N2',
 'N1',
 'N3',
 '<EOS>']